In [1]:
import insta_collector_loader
import importlib 
importlib.reload(insta_collector_loader)
from insta_collector_loader import InstaCollector, InstaLoader

#Install the 'ipython-sql' and 'prettytable' libraries using pip
!pip install ipython-sql prettytable

# Import necessary Python modules for API calls, JSON handling, SQLite, datetime, and data analysis
import requests, json, sqlite3, sys
from datetime import datetime
import pandas as pd
import numpy as np
import prettytable 
%matplotlib inline
prettytable.DEFAULT = 'DEFAULT'

# Load SQL magic extension to run SQL queries directly in notebook cells
%load_ext sql


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Instagram Data Collection & Preparation
> In this section, we set up a object-oriented pipeline to **collect raw post data and insights from the Instagram Graph API**, store them in a local SQLite database, and then **transform the JSON responses into a structured dataset** for analysis.

### API Credentials & Queries
>
>API credentials are securely stored in a local graph_api.txt file (not uploaded to GitHub).
The following queries are defined for use with the Graph API:
>- **Post fields**: post metadata (caption, timestamp, likes, comments, etc.)
>- **Feed insights**: reach, saves, shares, total interactions, follows, profile activity, views
>- **Reels insights**: reach, saves, shares, total interactions, views, average watch time, total watch time
>- **Account fields**: follower count, media count, follows count

In [10]:
# Load API credentials from local JSON file
with open('graph_api.txt', 'r') as file:
	info = json.loads(file.read())
	ACCESS_TOKEN = info['ACCESS_TOKEN']
	IG_BUSINESS_ID = info['IG_BUSINESS_ID']
	APP_ID = info['APP_ID']
	APP_SECRET = info['APP_SECRET']

# Define API queries for posts, insights, and account data
insta_queries = {
  'post_fields' : {
 	'access_token': ACCESS_TOKEN, # the access token must be used as a field for all API calls
    'fields' : 'id,media_type,media_url,username,comments_count,like_count,timestamp,caption,comments',
  },
  'feed_insights' : {
    'access_token': ACCESS_TOKEN,
    'metric' : 'reach,saved,shares,total_interactions,follows,profile_visits,profile_activity,views'
  },
  'reels_insights' : {
    'access_token': ACCESS_TOKEN,
    'metric' : 'reach,saved,shares,total_interactions,views,ig_reels_video_view_total_time,ig_reels_avg_watch_time'
  },
  'account_fields' : {
    'access_token' : ACCESS_TOKEN,
    'fields' : 'followers_count,media_count,follows_count'
  }
}

# Base URL and endpoints for API calls
base_url = "https://graph.facebook.com/v22.0/" # used for all API calls
endpoints = {
    'instagram_business_id': IG_BUSINESS_ID,
    'media': 'media',
    'insights': 'insights'
}

### Initialize Collector & Gather Data
>Set up the database, fetch post data and insights, and store them in SQLite.

In [ ]:
# Initialize the Instagram data collector and database
insta_collect = InstaCollector(ACCESS_TOKEN, IG_BUSINESS_ID, base_url, endpoints, insta_queries)
insta_collect.db_initializer(db = 'ig_data2.db')

# Uncomment if you want to reset tables (run only once or when resetting)
# insta_collect.db_tables()

# Collect posts data
insta_collect.get_fields()

# Collect post insights (metrics)
insta_collect.get_insights()

# Close DB connection
insta_collect.db_closer()

### Transform & Save Insights
>Parse and flatten raw JSON into a clean DataFrame, then save to SQLite.

In [ ]:
# Initialize data loader to transform raw JSON into structured data
insta_load = InstaLoader('ig_data2.db')

# Load posts fields from SQLite JSON to Python dicts
insta_load.fields_loader()

# Load insights JSON, flatten, and save to SQLite table 'insights'
insta_load.insights_loader()

# Close the DB connection
insta_load.db_closer()

## Extracting & Engineering Instagram Post Metrics
> - Connect to SQLite database
> - Connect to the local database `ig_data2.db` containing Instagram post JSON and insights data.
> - Initialize SQL Magic (`%sql`) to run queries directly from the notebook.

In [11]:
# Connect to the local SQLite database containing Instagram insights and post JSON data
db = 'ig_data2.db'
conn = sqlite3.connect(db)
cursor = conn.cursor()

# Initialize SQL Magic with database connection
%sql sqlite:///ig_data2.db

### Create SQL View
> **Step 1: Extract core post-level fields from JSON**
> - `media_type`: Type of post (image, video, reel)
> - `like_count`, `comments_count`: Engagement metrics
> - `timestamp`: Post timestamp (formatted)
> - `caption_length`: Number of characters in caption
> - `num_hashtags`: Number of hashtags in caption
>
> **Step 2: Join post fields with engagement and reach metrics**
> - Merge `posts` with `insights` table to include `reach`, `saved`, `shares`, `profile_activity`, etc.
>
> **Step 3: Add temporal features**
> - `days_since_last_post`: Gap since previous post
> - `days_since_first_post`: Gap since first post
> - `day_of_week` and `time_of_day`: Categorical time features for posting patterns
> 
> **Step 4: Compute engagement-related metrics**
> - `engagement`: Total likes + comments
> - `engagement_per_reach_pct`: Engagement relative to reach
> - `rolling_avg_engagement_3posts`: Rolling average engagement over last 3 posts
> - `engagement_rank_by_media_type`: Rank posts by engagement within each media type
> 
> **Step 5: Combine all features**
> - Join post-level fields, temporal features, and engagement metrics into a single SQL view
> - Order by most recent posts (`timestamp DESC`)

In [ ]:
%%sql 

-- Remove the view if it already exists
DROP VIEW IF EXISTS ig_post_metrics;

-- Create a view aggregating Instagram post data with engineered features
CREATE VIEW ig_post_metrics AS

-- Step 1: Extract core post-level fields from JSON
WITH posts AS (
    SELECT 
        post_id,
        json_extract(fields_json, '$.media_type') AS media_type, -- Type of post (e.g., image, video, reel)
        json_extract(fields_json, '$.like_count') AS like_count, -- Number of likes
        json_extract(fields_json, '$.comments_count') AS comments_count, -- Number of comments
        REPLACE(SUBSTR(json_extract(fields_json, '$.timestamp'), 1, 19), 'T', ' ') AS timestamp, -- Post timestamp (formatted)
        LENGTH(json_extract(fields_json, '$.caption')) AS caption_length, -- Total characters in caption
        LENGTH(json_extract(fields_json, '$.caption')) - LENGTH(
            REPLACE(json_extract(fields_json, '$.caption'), '#', '')) AS num_hashtags -- Count of hashtags
    FROM kgc_json
),

-- Step 2: Join post fields with engagement and reach metrics
posts_with_insights AS (
    SELECT
        p.*,
        i.reach, i.saved, i.shares, i.total_interactions,
        i.follows, i.profile_visits, i.profile_activity,
        i.views, i.ig_reels_avg_watch_time,
        i.ig_reels_video_view_total_time
    FROM 
        posts p
    LEFT JOIN 
        insights i ON i.id = p.post_id
),

-- Step 3: Add temporal features (posting patterns)
post_time_features AS (
    SELECT 
        post_id, 
        JULIANDAY(timestamp) - LAG(JULIANDAY(timestamp)) 
            OVER (ORDER BY timestamp) AS days_since_last_post, -- Gap since previous post
        JULIANDAY(timestamp) - JULIANDAY(FIRST_VALUE(timestamp)
            OVER(ORDER BY timestamp ASC)) AS days_since_first_post, -- Gap since first post in dataset        
        CASE 
            WHEN STRFTIME('%w', timestamp) = '0' THEN 'Sunday'
            WHEN STRFTIME('%w', timestamp) = '1' THEN 'Monday'
            WHEN STRFTIME('%w', timestamp) = '2' THEN 'Tuesday'
            WHEN STRFTIME('%w', timestamp) = '3' THEN 'Wednesday'
            WHEN STRFTIME('%w', timestamp) = '4' THEN 'Thursday'
            WHEN STRFTIME('%w', timestamp) = '5' THEN 'Friday'
            ELSE 'Saturday' 
        END AS day_of_week, -- Day of week posted
        CASE 
            WHEN CAST(STRFTIME('%H', timestamp) AS INT) BETWEEN 6 AND 11 THEN 'Morning'
            WHEN CAST(STRFTIME('%H', timestamp) AS INT) BETWEEN 12 AND 16 THEN 'Afternoon'
            WHEN CAST(STRFTIME('%H', timestamp) AS INT) BETWEEN 17 AND 20 THEN 'Evening'
            ELSE 'Night'
        END AS time_of_day -- Time of day posted
    FROM
        posts_with_insights
),

-- Step 4: Compute engagement-related metrics
post_engagement_metrics AS (
    SELECT 
        post_id, 
        like_count + comments_count AS engagement, -- Total engagements
        ROUND(
            (like_count + comments_count) * 100 / NULLIF(reach, 0),
            2) AS engagement_per_reach_pct, -- Engagement as % of reach
        ROUND(
            AVG(like_count + comments_count) 
            OVER (ORDER BY timestamp ROWS BETWEEN 2 PRECEDING AND CURRENT ROW))
            AS rolling_avg_engagement_3posts, -- Rolling average engagement
        ROUND(
            AVG(reach)
            OVER (ORDER BY timestamp ROWS BETWEEN 2 PRECEDING AND CURRENT ROW))
            AS rolling_avg_reach_3posts, -- Rolling average reach
        RANK() 
            OVER (PARTITION BY media_type ORDER BY (like_count + comments_count) DESC) 
            AS engagement_rank_by_media_type -- Group by media type, rank by post engagement
    FROM 
        posts_with_insights
)

-- Step 5: Combine all features into final view
SELECT 
    pwi.*,
    ptf.days_since_last_post,
    ptf.days_since_first_post,
    ptf.day_of_week,
    ptf.time_of_day,
    pem.engagement,
    pem.rolling_avg_engagement_3posts,
    pem.rolling_avg_reach_3posts,
    pem.engagement_per_reach_pct,
    pem.engagement_rank_by_media_type
FROM 
    posts_with_insights pwi
LEFT JOIN 
    post_time_features ptf ON pwi.post_id = ptf.post_id
LEFT JOIN 
    post_engagement_metrics pem ON pwi.post_id = pem.post_id
ORDER BY timestamp DESC

### Load SQL View Into DataFrame
>
>- Use `%sql` to query `ig_post_metrics` and convert results to a DataFrame (`ig_metrics_df`) for further analysis.
>- Once data is in pandas, we close the database connection.

In [ ]:
# Query the engineered SQL view into a pandas DataFrame for analysis
ig_post_metrics = %sql SELECT * FROM ig_post_metrics 
ig_metrics_df = ig_post_metrics.DataFrame()

# Close the SQLite connection (not needed once data is in DataFrame)
conn.close()

## Data Overview & Cleaning
>
>- Display the DataFrame structure, column data types, and non-null counts.
>- Generate summary statistics for both numeric and categorical columns.
>- Identify the number of missing values in each column.
>- Replace `NaN` values with 0 in selected columns (`caption_length`, `num_hashtags`, `days_since_last_post`) before analysis.

In [14]:
# Display summary of DataFrame structure, data types, and non-null counts
ig_metrics_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 26 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   post_id                         97 non-null     int64  
 1   media_type                      97 non-null     object 
 2   like_count                      97 non-null     int64  
 3   comments_count                  97 non-null     int64  
 4   timestamp                       97 non-null     object 
 5   caption_length                  89 non-null     float64
 6   num_hashtags                    89 non-null     float64
 7   reach                           35 non-null     float64
 8   saved                           35 non-null     float64
 9   shares                          35 non-null     float64
 10  total_interactions              35 non-null     float64
 11  follows                         24 non-null     float64
 12  profile_visits                  24 non

In [11]:
# Descriptive statistics for all columns (numeric & categorical)
ig_metrics_df.describe(include = 'all')

,post_id,media_type,like_count,comments_count,timestamp,caption_length,num_hashtags,reach,saved,shares,...,ig_reels_video_view_total_time,days_since_last_post,days_since_first_post,day_of_week,time_of_day,engagement,rolling_avg_engagement_3posts,rolling_avg_reach_3posts,engagement_per_reach_pct,engagement_rank_by_media_type
count,9.700000e+01,97,97.000000,97.000000,97,89.000000,89.000000,35.000000,35.000000,35.000000,...,1.100000e+01,96.000000,97.000000,97,97,97.000000,97.000000,35.000000,35.000000,97.000000
unique,NaN,3,NaN,NaN,97,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,7,3,NaN,NaN,NaN,NaN,NaN
top,NaN,CAROUSEL_ALBUM,NaN,NaN,2025-07-03 08:03:31,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Wednesday,Morning,NaN,NaN,NaN,NaN,NaN
freq,NaN,43,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,25,55,NaN,NaN,NaN,NaN,NaN
mean,1.802788e+16,NaN,22.226804,0.463918,NaN,105.078652,0.067416,851.571429,1.771429,2.028571,...,8.956386e+06,12.189742,538.334634,NaN,NaN,22.690722,22.432990,847.171429,3.000000,18.206186
std,1.332764e+14,NaN,18.365789,1.051406,NaN,128.244194,0.330212,681.669201,2.087578,2.255525,...,6.115310e+06,12.766512,381.005808,NaN,NaN,18.674493,12.448148,497.312689,2.363945,11.488163
min,1.784304e+16,NaN,1.000000,0.000000,NaN,1.000000,0.000000,255.000000,0.000000,0.000000,...,1.785540e+06,0.000197,0.000000,NaN,NaN,1.000000,1.000000,305.000000,1.000000,1.000000
25%,1.794300e+16,NaN,11.000000,0.000000,NaN,29.000000,0.000000,428.000000,0.000000,0.500000,...,4.138412e+06,3.063449,175.134699,NaN,NaN,12.000000,15.000000,500.500000,1.000000,9.000000
50%,1.799818e+16,NaN,17.000000,0.000000,NaN,55.000000,0.000000,658.000000,1.000000,1.000000,...,7.947541e+06,10.998166,542.218206,NaN,NaN,17.000000,19.000000,553.000000,2.000000,16.000000
75%,1.807488e+16,NaN,26.000000,0.000000,NaN,123.000000,0.000000,1082.500000,2.000000,3.000000,...,1.325185e+07,15.984725,863.788877,NaN,NaN,28.000000,29.000000,1150.500000,4.000000,27.000000


In [15]:
# Check the number of missing values in each column of the DataFrame
ig_metrics_df.isnull().sum()

post_id                            0
media_type                         0
like_count                         0
comments_count                     0
timestamp                          0
caption_length                     8
num_hashtags                       8
reach                             62
saved                             62
shares                            62
total_interactions                62
follows                           73
profile_visits                    73
profile_activity                  73
views                             62
ig_reels_avg_watch_time           86
ig_reels_video_view_total_time    86
days_since_last_post               1
days_since_first_post              0
day_of_week                        0
time_of_day                        0
engagement                         0
rolling_avg_engagement_3posts      0
rolling_avg_reach_3posts          62
engagement_per_reach_pct          62
engagement_rank_by_media_type      0
dtype: int64

In [16]:
# Replace NaN values with 0 in selected columns to handle missing data before analysis
cols = ['caption_length', 'num_hashtags', 'days_since_last_post']
ig_metrics_df[cols] = ig_metrics_df[cols].fillna(0)

## Export Final Metrics Table 
> 1. **Export DataFrame to CSV**: Save the cleaned and processed DataFrame to a CSV file without including the index.
> 2. **Preview the Final Table**: Display the first ten rows of the DataFrame

In [4]:
# Save DataFrame to CSV file
ig_metrics_df.to_csv('ig_post_metrics.csv', index = False)

# View the final table
ig_metrics_df.head(10)

,post_id,media_type,like_count,comments_count,timestamp,caption_length,num_hashtags,reach,saved,shares,...,ig_reels_video_view_total_time,days_since_last_post,days_since_first_post,day_of_week,time_of_day,engagement,rolling_avg_engagement_3posts,rolling_avg_reach_3posts,engagement_per_reach_pct,engagement_rank_by_media_type
0,18055341356588190,IMAGE,21,0,2025-07-03 08:03:31,16.0,0.0,579.0,0.0,1.0,...,NaN,17.066019,1170.215278,Thursday,Morning,21,43.0,1041.0,3.0,10
1,17924390040086729,IMAGE,25,3,2025-06-16 06:28:27,47.0,0.0,766.0,1.0,2.0,...,NaN,11.791157,1153.149259,Monday,Morning,28,43.0,1237.0,3.0,4
2,17917655700098162,VIDEO,78,2,2025-06-04 11:29:11,17.0,0.0,1777.0,5.0,3.0,...,17981036.0,5.078438,1141.358102,Wednesday,Morning,80,40.0,1316.0,4.0,2
3,18109512208443384,IMAGE,18,2,2025-05-30 09:36:14,43.0,0.0,1169.0,5.0,2.0,...,NaN,0.002847,1136.279664,Friday,Morning,20,23.0,1246.0,1.0,11
4,18114907681473883,IMAGE,17,3,2025-05-30 09:32:08,9.0,0.0,1003.0,0.0,3.0,...,NaN,11.089838,1136.276817,Friday,Morning,20,35.0,2128.0,1.0,11
5,18176836600325450,IMAGE,24,4,2025-05-19 07:22:46,16.0,0.0,1565.0,2.0,6.0,...,NaN,16.984630,1125.186979,Monday,Morning,28,40.0,2044.0,1.0,4
6,17976668054839065,VIDEO,58,0,2025-05-02 07:44:54,33.0,0.0,3815.0,10.0,11.0,...,19048924.0,4.300035,1108.202350,Friday,Morning,58,39.0,2017.0,1.0,4
7,17978602919829685,VIDEO,31,2,2025-04-28 00:32:51,52.0,0.0,753.0,3.0,0.0,...,11630355.0,11.879954,1103.902315,Monday,Night,33,25.0,1036.0,4.0,6
8,17972006636849182,VIDEO,24,2,2025-04-16 03:25:43,45.0,0.0,1483.0,0.0,4.0,...,5374370.0,13.618125,1092.022361,Wednesday,Night,26,18.0,1173.0,1.0,9
9,18088780102606381,IMAGE,11,4,2025-04-02 12:35:37,15.0,0.0,873.0,4.0,2.0,...,NaN,15.135903,1078.404236,Wednesday,Afternoon,15,18.0,1233.0,1.0,16
